# 01 — Data Exploration & Non-IID CIFAR-10 Split

Validates the shard-based non-IID data split used across all experiments.
- Visualise class distribution per node
- Confirm shards_per_node=2 creates highly heterogeneous partitions
- Sanity-check data loaders

In [ ]:
import subprocess, sys, os

repo_root = os.getcwd()
if not os.path.isdir(os.path.join(repo_root, "src")):
    repo_root = os.path.dirname(repo_root)

assert os.path.isdir(os.path.join(repo_root, "src")), \
    f"src/ not found. cwd={os.getcwd()} — open this notebook from the fl-rockstar repo root"

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

subprocess.check_call([sys.executable, "-m", "pip", "install",
    "torch", "torchvision", "datasets", "networkx",
    "numpy", "scipy", "matplotlib", "seaborn", "pandas",
    "tqdm", "pyyaml", "--quiet"])

import torch
import numpy as np
import matplotlib.pyplot as plt
from src.data import make_noniid_cifar10

device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available()
          else "cpu")
print(f"device: {device}  |  root: {repo_root}")

In [ ]:
N_NODES = 100
node_loaders, test_loader = make_noniid_cifar10(
    n_nodes=N_NODES, shards_per_node=2, batch_size=32
)
print(f'Nodes: {len(node_loaders)} | Test batches: {len(test_loader)}')

In [ ]:
CLASSES = ['airplane','automobile','bird','cat','deer',
           'dog','frog','horse','ship','truck']

# Sample 10 random nodes
sample_nodes = [0, 5, 10, 20, 30, 40, 50, 60, 75, 99]
dist_matrix = np.zeros((len(sample_nodes), 10))

for row, node_id in enumerate(sample_nodes):
    counts = [0] * 10
    for X, y in node_loaders[node_id]:
        for label in y:
            counts[label.item()] += 1
    dist_matrix[row] = counts

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(dist_matrix, aspect='auto', cmap='Blues')
ax.set_xticks(range(10))
ax.set_xticklabels(CLASSES, rotation=30, ha='right')
ax.set_yticks(range(len(sample_nodes)))
ax.set_yticklabels([f'Node {n}' for n in sample_nodes])
ax.set_title('Class distribution per node (shards_per_node=2, Non-IID)')
plt.colorbar(im, ax=ax, label='sample count')
plt.tight_layout()
plt.savefig('../results/01_data_distribution.png', dpi=150)
plt.show()
print('Each row should have only 2 non-zero columns → confirmed Non-IID')

In [ ]:
total_train = sum(
    sum(len(y) for _, y in loader)
    for loader in node_loaders.values()
)
total_test = sum(len(y) for _, y in test_loader)
print(f'Total training samples across all nodes: {total_train:,}')
print(f'Total test samples: {total_test:,}')
print(f'Average per node: {total_train // N_NODES}')